# YouTube to Bilibili - Google Colab 免服务器部署环境

这个笔记本适用于在 Google Colab 免费提供的临时实例（CPU 即可）上运行该项目，并通过 **Cloudflare Tunnel** 生成公网访问链接。这就意味着**你不需要有自己的服务器**，也没有额度受限问题，同时直接在海外网络环境下使用极速下载并处理 YouTube 视频。

### 📖 使用说明
由于你需要依托 GitHub 代码库拉取项目，仅需**按顺序执行下方三个代码块**（点击每个代码块左侧的 ▶️ 播放按钮），一切都会全自动配置。等到第三个代码块执行完毕，会生成专有的外网可视接口（链接）。

In [ ]:
# 1. 自动准备系统环境并拉取项目代码
import os

!apt-get update -qq > /dev/null
!apt-get install -qq -y ffmpeg git curl unzip > /dev/null

# 下载并安装 Deno 环境 (yt-dlp 处理部分含JS验证的视频必备)
!curl -fsSL https://deno.land/install.sh | sh > /dev/null

# =========================================================
# GitHub 仓库地址，如果你的地址有变，请在这里修改为您实际的仓库 URL
# =========================================================
REPO_URL = "https://github.com/eleven-monkey/yt-to-bili.git"
REPO_NAME = REPO_URL.split('/')[-1].replace('.git', '')

# 自动判断并处理代码仓库克隆或更新
if not os.path.exists(REPO_NAME):
    print(f"🔗 正在克隆项目仓库: {REPO_URL} ...")
    !git clone -q {REPO_URL}
else:
    print(f"✅ 项目目录 '{REPO_NAME}' 已存在，尝试拉取最新更新...")
    %cd {REPO_NAME}
    !git pull -q
    %cd ..

# 切换至项目内，安装所需 Python 依赖等
os.chdir(REPO_NAME)
print("📦 正在安装 Python 依赖包...")
!pip install -q -r requirements.txt

print("🎉 第一步环境准备完成！")

In [ ]:
# 2. 下载并配置文件穿透所需工具 (Cloudflared Tunnel 轻量版)
import os

if not os.path.exists("cloudflared-linux-amd64"):
    print("⬇️ 正在下载并配置 Cloudflare 内网穿透服务工具...")
    !wget -q -O cloudflared-linux-amd64 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared-linux-amd64
    print("✅ Cloudflare Tunnel 安装完成。")
else:
    print("✅ Cloudflare Tunnel 已经存在，无需操作。")

In [ ]:
# 3. 一键启动服务，并生成公网访问连接
import subprocess
import time
import os
import re

# 确保系统的 PATH 包含我们第一步下载的 Deno 环境
os.environ["PATH"] = f"/root/.deno/bin:{os.environ.get('PATH', '')}"

print("🚀 正在后台启动应用进程...")
# 将 Streamlit 绑定至后台默认 8501 端口而不阻塞主界面
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=127.0.0.1"],
    stdout=subprocess.DEVNULL, 
    stderr=subprocess.STDOUT
)

time.sleep(4)  # 稍微等待应用加载就绪

print("🌐 正在请求公网访问链接...")
tunnel_process = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = ""
# 实时抓取命令输出并使用正则表达式提取 trycloudflare.com 链接内容
# 使用正则可以有效避免控制台 ANSI 转义字符(\\033 等)导致的截断或前后杂乱字符问题
for line in tunnel_process.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "=*=" * 20)
    print("🎉 服务启动成功！请点击下方专属公网链接访问应用：")
    print(f"\n    ▶️ {public_url} ◀️\n")
    print("=*=" * 20 + "\n")
    print("⚠️ 极重要提示：由于 Colab 对非活跃进程会有休眠机制，使用期间，请保此网页的代码处于【正在运行状态】，不可以关闭本网页或关闭你的电脑。")
    print("💡 若需强制中止重新启动，可点击执行单元的停止符号（或按下 Ctrl+M+I）。")
else:
    print("❌ 连接穿透服务时出错，请尝试重新运行本代码块。")

try:
    # 堵塞，保证进程不退出
    tunnel_process.wait()
except KeyboardInterrupt:
    print("\n🛑 检测到你的主动中断操作，正在关闭后台 Web 服务与穿透进程...")
    streamlit_process.kill()
    tunnel_process.kill()
    print("👋 服务已完全关闭。")